In [10]:
from kilosort_pipeline.utils import load_config, parse_openephys_folders
from kilosort_pipeline.sync import match_chirp_edges
import re
import spikeinterface.extractors as se

from pathlib import Path
from collections import defaultdict
from loguru import logger
import numpy as np
import pynapple as nap
from scipy.interpolate import make_interp_spline

def log_ts(timestamps, name):
    start_ts = timestamps[0]
    end_ts = timestamps[-1]
    logger.info(f"{name}: {start_ts:.4f} ... {end_ts:.4f} s ({timestamps.size} samples)")

def load_events(events_path, cont_path):
    event_ts  = np.load(events_path, mmap_mode='r')
    cont_ts   = np.load(cont_path, mmap_mode='r')
    states    = np.load(events_path.replace('timestamps.npy', 'states.npy'), mmap_mode='r')
    return event_ts, cont_ts, states

def get_kilosort_spikes(output_path, probe_filter=None):
    spike_times_dict = {}

    kilosort_files = list(Path(output_path).glob('*/kilosort/spike_times.npy'))
    if not kilosort_files:
        logger.error("No Kilosort output found. Run Kilosort first.")
        raise FileNotFoundError(f"No spike_times.npy files in {output_path}")
    
    for spike_file in kilosort_files:
        probe_name = spike_file.parent.parent.name

        # Filter probes if requested
        if probe_filter and probe_name not in probe_filter:
            continue

        spike_times_dict[probe_name] = np.load(spike_file, mmap_mode='r')
        logger.info(f"Loaded {len(spike_times_dict[probe_name])} spikes from {probe_name}")

    return spike_times_dict

In [2]:
paths = ['/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField',
  '/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField']

probe_filter = ['ProbeA', 'ProbeB']

ps = parse_openephys_folders(paths, probe_filter)

2025-11-01 17:49:41.332 | INFO     | kilosort_pipeline.utils:parse_openephys_folders:158 - Parsing OpenEphys folders
2025-11-01 17:49:41.523 | DEBUG    | kilosort_pipeline.utils:parse_openephys_folders:162 - Session 1/4: AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-01 17:49:50.492 | DEBUG    | kilosort_pipeline.utils:parse_openephys_folders:190 -   OneBox-ADC: 1 segment(s), 1 event files, 1 cont files
2025-11-01 17:49:54.352 | DEBUG    | kilosort_pipeline.utils:parse_openephys_folders:190 -   ProbeA: 1 segment(s), 1 event files, 1 cont files
2025-11-01 17:49:58.086 | DEBUG    | kilosort_pipeline.utils:parse_openephys_folders:190 -   ProbeB: 1 segment(s), 1 event files, 1 cont files
2025-11-01 17:49:58.125 | DEBUG    | kilosort_pipeline.utils:parse_openephys_folders:162 - Session 2/4: AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField
2025-11-01 17:50:06.792 | DEBUG    | kilosort_pipeline.utils:parse_openephys_folders:190 -   OneBox-ADC: 1 segment(s), 2 event files, 2 cont

In [6]:
ks_spikes = get_kilosort_spikes(output_path='/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/kilosort_output/AA001_Day2', probe_filter=['ProbeA', 'ProbeB'])
ks_spikes

2025-11-01 17:55:17.989 | INFO     | __main__:get_kilosort_spikes:34 - Loaded 164597833 spikes from ProbeB
2025-11-01 17:55:18.507 | INFO     | __main__:get_kilosort_spikes:34 - Loaded 110944227 spikes from ProbeA


{'ProbeB': memmap([        11,         13,         17, ..., 1027373483, 1027373486,
         1027373494], shape=(164597833,)),
 'ProbeA': memmap([         4,         14,         14, ..., 1010766696, 1010766697,
         1010766703], shape=(110944227,))}

In [28]:
class Timestamps:
    def __init__(self, name, fs, t_start=0.0):
        self.name = name
        self.fs = fs
        self.dt = 1 / fs
        
        self.sync_timestamps = []
        self.intervals = []
        self.t_offset = t_start
        self.starting_states = []

    def update(self, sync_ts, t_end, starting_state=None):
        # Update synchronization timestamps
        global_event_ts = sync_ts + self.t_offset
        self.sync_timestamps.append(global_event_ts)

        # Update intervals
        global_cont_start = self.t_offset
        global_cont_end = self.t_offset + t_end
        self.intervals.append((global_cont_start, global_cont_end))
        
        # Update offset for next segment
        self.t_offset = global_cont_end + self.dt

        if starting_state:
            self.starting_states.append(starting_state)
        
    def __repr__(self):
        return f"Timestamps(name='{self.name}', segments={self.num_segments()}, fs={self.fs:.1f} Hz)"

In [29]:
ADC = Timestamps(name='OneBox-ADC', fs=30300.0, t_start=0.0)

adc_event_paths = ps["timestamps"]["OneBox-ADC"]['event']
adc_cont_paths = ps["timestamps"]["OneBox-ADC"]['cont']

for idx, (event_path, cont_path) in enumerate(zip(adc_event_paths, adc_cont_paths)):
    event_ts, cont_ts, states = load_events(event_path, cont_path)

    # Subtract the offset of continuous recording
    ADC.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0], starting_state=states[0])

    ########## LOGGING #################################
    log_ts(event_ts, "ADC event")
    log_ts(cont_ts, "ADC cont")
    logger.info(f"ADC interval: {ADC.intervals[-1][0]:.4f} ... {ADC.intervals[-1][1]:.4f} s")
    log_ts(ADC.sync_timestamps[-1], "ADC global segment")
    log_ts(np.concatenate(ADC.sync_timestamps), "ADC global")
    logger.info("-"*60)

2025-11-01 18:56:51.186 | INFO     | __main__:log_ts:16 - ADC event: 12.0830 ... 3645.1092 s (7267 samples)
2025-11-01 18:56:51.187 | INFO     | __main__:log_ts:16 - ADC cont: 11.6334 ... 3645.4310 s (110114290 samples)
2025-11-01 18:56:51.188 | INFO     | __main__:<module>:15 - ADC interval: 0.0000 ... 3633.7977 s
2025-11-01 18:56:51.188 | INFO     | __main__:log_ts:16 - ADC global segment: 0.4497 ... 3633.4758 s (7267 samples)
2025-11-01 18:56:51.189 | INFO     | __main__:log_ts:16 - ADC global: 0.4497 ... 3633.4758 s (7267 samples)
2025-11-01 18:56:51.189 | INFO     | __main__:<module>:18 - ------------------------------------------------------------
2025-11-01 18:56:51.974 | INFO     | __main__:log_ts:16 - ADC event: 123.0811 ... 2266.3990 s (11788 samples)
2025-11-01 18:56:51.975 | INFO     | __main__:log_ts:16 - ADC cont: 123.0448 ... 2266.4321 s (64945708 samples)
2025-11-01 18:56:51.976 | INFO     | __main__:<module>:15 - ADC interval: 3633.7977 ... 5777.1850 s
2025-11-01 18:56

In [ ]:
probe_filter = ['ProbeA', 'ProbeB']
probe_timestamps = {k:d for k,d in ps["timestamps"].items() if k != "OneBox-ADC" and k in probe_filter}
synced_spikes = []

masks = {}
for probe, paths in probe_timestamps.items():
    PRB = Timestamps(name=probe, fs=30000.0, t_start=0.0)
    logger.info(f"Processing probe: {probe}")

    logger.info("Extracting kilosort spikes")
    kilosort_spikes = ks_spikes[probe] / PRB.fs
    log_ts(kilosort_spikes, f"{probe} spikes")
    total_spikes_left = kilosort_spikes.size
    logger.info('='*60)

    masks[probe] = []
    for idx, (ev_path, cont_path) in enumerate(zip(paths['event'], paths['cont'])):
        event_ts, cont_ts, states = load_events(ev_path, cont_path)

        # Handle state mismatches
        if states[0] != ADC.starting_states[idx]:
            logger.warning(f"State mismatch between {probe} and ADC")
            logger.info("Matching edges")
            event_ts, _ = match_chirp_edges(event_ts, ADC.sync_timestamps(idx))

        # Update probe timestamps, starting state is not needed here
        PRB.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0])
        probe_times = PRB.sync_timestamps[-1]
        ########## LOGGING #################################
        log_ts(event_ts, f"Event timestamps")
        log_ts(cont_ts, f"Continuous timestamps")
        log_ts(probe_times, f"Global segment")
        log_ts(np.concatenate(PRB.sync_timestamps), f"Global")
        ########## LOGGING #################################

        adc_times = ADC.sync_timestamps(idx)
        
        # Extract spikes based on continuous range
        cont_start, cont_end = PRB.intervals[idx]
        mask = (kilosort_spikes >= cont_start) & (kilosort_spikes <= cont_end)
        masks[probe].append(mask) # DEBUG purpose
        
        probe_spikes = kilosort_spikes[mask]
        log_ts(probe_spikes, f"Extracted spikes")
        
        # Handle length mismatches
        min_length = min(len(probe_times), len(adc_times))
        if min_length < len(adc_times):
            logger.warning(f"  Truncating ADC timestamps. ADC timestamps: {len(adc_times)} -> {min_length}.")
            adc_times = adc_times[:min_length]
        elif min_length < len(probe_times):
            logger.warning(f"  Truncating Probe timestamps. Probe timestamps: {len(probe_times)} -> {min_length}.")
            probe_times = probe_times[:min_length]
        
        # Interpolate/extrapolate to ADC time
        spl = make_interp_spline(x=probe_times, y=adc_times, k=1)
        adc_spikes = spl(probe_spikes)
        synced_spikes.append(adc_spikes)
        total_spikes_left -= adc_spikes.size

        log_ts(adc_spikes, "ADC interpolated spikes")
        logger.info(f"Synced spikes: {adc_spikes.size}/{kilosort_spikes.size}. Remaining spikes: {total_spikes_left}")
        logger.info("="*60)

2025-11-01 18:56:54.353 | INFO     | __main__:<module>:8 - Processing probe: ProbeA
2025-11-01 18:56:54.354 | INFO     | __main__:<module>:10 - Extracting kilosort spikes
